# Assignment 5 (Part B) — LLM Prompt Engineering: Natural Language → SQL (Multi-table)



## 1) Define the database schema

### Tables
**Customers**
- customer_id (PK)
- name
- city

**Orders**
- order_id (PK)
- customer_id (FK → Customers.customer_id)
- order_date
- total_amount

**OrderItems**
- order_item_id (PK)
- order_id (FK → Orders.order_id)
- product_name
- quantity
- unit_price


## 2) Create a tiny SQLite database (optional validation)

In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL
);

CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY(customer_id) REFERENCES Customers(customer_id)
);

CREATE TABLE OrderItems (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY(order_id) REFERENCES Orders(order_id)
);
""")

cur.executemany("INSERT INTO Customers VALUES (?, ?, ?)", [
    (1, "Ava Patel", "Boston"),
    (2, "Noah Kim", "Seattle"),
    (3, "Mia Chen", "Boston"),
])

cur.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?)", [
    (101, 1, "2026-01-10", 120.50),
    (102, 1, "2026-01-20", 75.00),
    (103, 2, "2026-01-05", 220.00),
    (104, 3, "2026-01-25", 40.00),
])

cur.executemany("INSERT INTO OrderItems VALUES (?, ?, ?, ?, ?)", [
    (1001, 101, "Keyboard", 1, 50.00),
    (1002, 101, "Mouse", 1, 20.50),
    (1003, 102, "USB-C Cable", 3, 5.00),
    (1004, 103, "Monitor", 1, 220.00),
    (1005, 104, "Notebook", 4, 2.50),
])

conn.commit()

pd.read_sql_query("SELECT * FROM Customers", conn)


,customer_id,name,city
0,1,Ava Patel,Boston
1,2,Noah Kim,Seattle
2,3,Mia Chen,Boston


## 3) Natural language query

**NL Query:**  
> "Show each Boston customer’s name and their total spending across all orders, sorted by total spending descending."

This requires joining **Customers** and **Orders** (2 tables).


## 4) Zero-shot prompting (manual)

```text
You are an expert SQL developer.

Database schema:
Customers(customer_id PK, name, city)
Orders(order_id PK, customer_id FK->Customers.customer_id, order_date, total_amount)
OrderItems(order_item_id PK, order_id FK->Orders.order_id, product_name, quantity, unit_price)

Task:
Translate the following natural language query into SQL.

Natural language query:
"Show each Boston customer’s name and their total spending across all orders, sorted by total spending descending."

Rules:
- Use standard SQL
- Use explicit JOINs
- Return only the SQL query (no explanation)
```

In [21]:
zero_shot_sql = """
SELECT
    c.name,
    COALESCE(SUM(o.total_amount), 0) AS total_spending
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE c.city = 'Boston'
GROUP BY c.name
ORDER BY total_spending DESC;
"""

print(zero_shot_sql)



SELECT
    c.name,
    COALESCE(SUM(o.total_amount), 0) AS total_spending
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE c.city = 'Boston'
GROUP BY c.name
ORDER BY total_spending DESC;



## 5) Few-shot prompting (manual)

```text
You are an expert SQL developer.

Database schema:
Customers(customer_id PK, name, city)
Orders(order_id PK, customer_id FK->Customers.customer_id, order_date, total_amount)
OrderItems(order_item_id PK, order_id FK->Orders.order_id, product_name, quantity, unit_price)

Examples:

Example 1 (NL):
"List all customers in Boston."
Example 1 (SQL):
SELECT customer_id, name
FROM Customers
WHERE city = 'Boston';

Example 2 (NL):
"Show each customer and the number of orders they placed."
Example 2 (SQL):
SELECT c.customer_id, c.name, COUNT(o.order_id) AS num_orders
FROM Customers c
LEFT JOIN Orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name;

Task:
Translate the following natural language query into SQL.

Natural language query:
"Show each Boston customer’s name and their total spending across all orders, sorted by total spending descending."

Rules:
- Use standard SQL
- Use explicit JOINs
- Use GROUP BY where needed
- Return only the SQL query (no explanation)
```

### Paste Few-shot SQL here

In [22]:
few_shot_sql = """
SELECT
    c.name,
    SUM(o.total_amount) AS total_spending
FROM Customers c
JOIN Orders o
    ON o.customer_id = c.customer_id
WHERE c.city = 'Boston'
GROUP BY c.customer_id, c.name
ORDER BY total_spending DESC;
"""

print(few_shot_sql)



SELECT
    c.name,
    SUM(o.total_amount) AS total_spending
FROM Customers c
JOIN Orders o
    ON o.customer_id = c.customer_id
WHERE c.city = 'Boston'
GROUP BY c.customer_id, c.name
ORDER BY total_spending DESC;



## 7) Validate generated SQL by executing it

In [23]:
def run_sql(sql: str):
    sql = sql.strip()
    if not sql or "PASTE" in sql.upper():
        raise ValueError("Paste a real SQL query string first.")
    return pd.read_sql_query(sql, conn)

for label, sql in [
    ("Zero-shot", zero_shot_sql),
    ("Few-shot", few_shot_sql),
]:
    try:
        df = run_sql(sql)
        print(f"\n=== {label} result ===")
        display(df)
    except Exception as e:
        print(f"\n=== {label} failed ===")
        print(e)



=== Zero-shot result ===


,name,total_spending
0,Ava Patel,195.5
1,Mia Chen,40.0



=== Few-shot result ===


,name,total_spending
0,Ava Patel,195.5
1,Mia Chen,40.0
